In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: Install Dependencies
# ═══════════════════════════════════════════════════════════════

!pip install -q transformers datasets kaggle nltk nlpaug scikit-learn
!pip install --upgrade transformers

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Import Libraries
# ═══════════════════════════════════════════════════════════════

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries
import nltk
import nlpaug.augmenter.word as naw
from nltk.corpus import stopwords

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

# TensorFlow & Transformers
import tensorflow as tf
from transformers import (
    DistilBertTokenizer,
    TFDistilBertForSequenceClassification,
    DistilBertConfig
)

# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("✓ All libraries imported successfully")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

ImportError: cannot import name 'TFDistilBertForSequenceClassification' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: Kaggle API Setup & Dataset Download
# ═══════════════════════════════════════════════════════════════

# Upload your kaggle.json file when prompted
from google.colab import files
print("Please upload your kaggle.json file:")
uploaded = files.upload()

# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d selener/consumer-complaint-database

print("✓ Dataset downloaded successfully")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Extract Dataset
# ═══════════════════════════════════════════════════════════════

!unzip -q consumer-complaint-database.zip
print("✓ Dataset extracted successfully")

# List files
!ls -lh

In [ ]:
# Load dataset
df = pd.read_csv('complaints.csv', low_memory=False)

print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nFirst Few Rows:")
print(df.head())

# Check for complaint text column
text_column = 'Consumer complaint narrative'
label_column = 'Product'

print(f"\nText Column: {text_column}")
print(f"Label Column: {label_column}")

# Remove rows with missing text or labels
df = df[[text_column, label_column]].dropna()
print(f"\nDataset after removing NaN: {df.shape}")

# Show class distribution
print("\n" + "="*60)
print("CLASS DISTRIBUTION")
print("="*60)
class_counts = df[label_column].value_counts()
print(class_counts)

# Visualize class distribution
plt.figure(figsize=(12, 6))
class_counts.plot(kind='barh')
plt.title('Number of Complaints per Product Category')
plt.xlabel('Count')
plt.ylabel('Product')
plt.tight_layout()
plt.show()

print(f"\nTotal Classes: {df[label_column].nunique()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: Data Balancing - Equal Sampling
# ═══════════════════════════════════════════════════════════════

# Select top 10 classes for faster training
top_classes = df[label_column].value_counts().head(10).index.tolist()
df_filtered = df[df[label_column].isin(top_classes)]

print(f"Selected Classes: {len(top_classes)}")
print(top_classes)

# Balance classes - sample equal number from each
SAMPLES_PER_CLASS = 5000  # Adjust based on smallest class

balanced_dfs = []
for product in top_classes:
    class_df = df_filtered[df_filtered[label_column] == product]
    if len(class_df) >= SAMPLES_PER_CLASS:
        sampled = class_df.sample(n=SAMPLES_PER_CLASS, random_state=42)
    else:
        # Oversample if class has fewer samples
        sampled = class_df.sample(n=SAMPLES_PER_CLASS, replace=True, random_state=42)
    balanced_dfs.append(sampled)

df_balanced = pd.concat(balanced_dfs, ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced Dataset Shape: {df_balanced.shape}")
print("\nBalanced Class Distribution:")
print(df_balanced[label_column].value_counts())


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: Text Preprocessing
# ═══════════════════════════════════════════════════════════════

def clean_text(text):
    """Clean and preprocess text"""
    if not isinstance(text, str):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove extra whitespace
    text = ' '.join(text.split())

    # Limit length (BERT max is 512 tokens)
    words = text.split()
    if len(words) > 200:
        text = ' '.join(words[:200])

    return text

# Apply cleaning
df_balanced['cleaned_text'] = df_balanced[text_column].apply(clean_text)

# Remove any empty texts
df_balanced = df_balanced[df_balanced['cleaned_text'].str.len() > 10]

print(f"Dataset after cleaning: {df_balanced.shape}")
print("\nSample cleaned text:")
print(df_balanced['cleaned_text'].iloc[0])


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: Data Augmentation (10× increase)
# ═══════════════════════════════════════════════════════════════

print("Starting data augmentation...")
print("This will take 5-10 minutes...")

# Create augmentation pipeline
aug = naw.SynonymAug(aug_src='wordnet', aug_max=3)

def augment_text(text, n_augments=3):
    """Generate augmented versions of text"""
    augmented = [text]  # Original text

    for _ in range(n_augments):
        try:
            aug_text = aug.augment(text)
            if isinstance(aug_text, list):
                augmented.extend(aug_text)
            else:
                augmented.append(aug_text)
        except:
            continue

    return augmented[:n_augments + 1]  # Return original + n_augments

# Augment a subset for faster processing (use 2× instead of 10× for 1-hour constraint)
AUGMENT_MULTIPLIER = 2

augmented_data = []
for idx, row in df_balanced.iterrows():
    augmented_texts = augment_text(row['cleaned_text'], n_augments=AUGMENT_MULTIPLIER-1)

    for aug_text in augmented_texts:
        augmented_data.append({
            'text': aug_text,
            'label': row[label_column]
        })

    if (idx + 1) % 5000 == 0:
        print(f"Processed {idx + 1}/{len(df_balanced)} samples")

df_augmented = pd.DataFrame(augmented_data)
df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✓ Augmentation Complete!")
print(f"Original size: {len(df_balanced)}")
print(f"Augmented size: {len(df_augmented)}")
print(f"Increase factor: {len(df_augmented) / len(df_balanced):.2f}×")


Starting data augmentation...
This will take 5-10 minutes...


Streaming output truncated to the last 5000 lines.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_dat

Processed 5000/49998 samples


Streaming output truncated to the last 5000 lines.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_dat

Processed 10000/49998 samples


...

Processed 40000/49998 samples


Streaming output truncated to the last 5000 lines.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_dat

Processed 45000/49998 samples


Streaming output truncated to the last 5000 lines.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_dat

Processed 50000/49998 samples

✓ Augmentation Complete!
Original size: 49998
Augmented size: 49998
Increase factor: 1.00×


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9: Label Encoding
# ═══════════════════════════════════════════════════════════════

label_encoder = LabelEncoder()
df_augmented['label_encoded'] = label_encoder.fit_transform(df_augmented['label'])

print("Label Mapping:")
for idx, label in enumerate(label_encoder.classes_):
    print(f"{idx}: {label}")

num_classes = len(label_encoder.classes_)
print(f"\nTotal Classes: {num_classes}")





Label Mapping:
0: Bank account or service
1: Checking or savings account
2: Consumer Loan
3: Credit card
4: Credit card or prepaid card
5: Credit reporting
6: Credit reporting, credit repair services, or other personal consumer reports
7: Debt collection
8: Mortgage
9: Student loan

Total Classes: 10


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10: Train-Validation Split (80-20)
# ═══════════════════════════════════════════════════════════════

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_augmented['text'].tolist(),
    df_augmented['label_encoded'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_augmented['label_encoded']
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 11: Tokenization (Memory-Efficient)
# ═══════════════════════════════════════════════════════════════

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 128  # Reduced for memory efficiency
BATCH_SIZE = 16   # Adjust based on GPU memory

print("Tokenizing training data...")
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors='tf'
)

print("Tokenizing validation data...")
val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors='tf'
)

print("✓ Tokenization complete")


In [ ]:

# ═══════════════════════════════════════════════════════════════
# CELL 12: Create TensorFlow Datasets (Memory-Safe)
# ═══════════════════════════════════════════════════════════════

train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
))

val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    val_labels
))

# Optimize dataset pipeline
train_dataset = train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("✓ TensorFlow datasets created")

# Clear memory
del train_encodings, val_encodings, df_augmented, df_balanced, df_filtered, df
gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 13: Model Architecture - DistilBERT
# ═══════════════════════════════════════════════════════════════

# Load pre-trained model
model = TFDistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes
)

print("✓ Model loaded successfully")
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 14: Model Compilation
# ═══════════════════════════════════════════════════════════════

optimizer = tf.keras.optimizers.Adam(learning_rate=3e-5)

loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=['accuracy']
)

print("✓ Model compiled")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 15: Callbacks
# ═══════════════════════════════════════════════════════════════

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    )
]

print("✓ Callbacks configured")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 16: Model Training
# ═══════════════════════════════════════════════════════════════

EPOCHS = 5  # Sufficient for transfer learning

print("Starting training...")
print("="*60)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Training complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 17: Training Visualization
# ═══════════════════════════════════════════════════════════════

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', marker='s')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Validation Loss', marker='s')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
print("\n" + "="*60)
print("FINAL TRAINING RESULTS")
print("="*60)
print(f"Final Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"Best Validation Accuracy: {max(history.history['val_accuracy']):.4f}")

In [ ]:

# ═══════════════════════════════════════════════════════════════
# CELL 18: Model Evaluation
# ═══════════════════════════════════════════════════════════════

print("Evaluating model on validation set...")

# Get predictions
predictions = model.predict(val_dataset)
predicted_labels = np.argmax(predictions.logits, axis=1)

# Calculate accuracy
val_accuracy = accuracy_score(val_labels, predicted_labels)
print(f"\n✓ Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")

# Classification report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(
    val_labels,
    predicted_labels,
    target_names=label_encoder.classes_,
    digits=4
))

# Confusion matrix
cm = confusion_matrix(val_labels, predicted_labels)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:

# ═══════════════════════════════════════════════════════════════
# CELL 19: Model Saving (Multiple Formats)
# ═══════════════════════════════════════════════════════════════

# Save in Keras format
model.save('complaint_classifier_model.keras')
print("✓ Model saved as: complaint_classifier_model.keras")

# Save weights in H5 format
model.save_weights('complaint_classifier_weights.h5')
print("✓ Weights saved as: complaint_classifier_weights.h5")

# Save using SavedModel format (for deployment)
model.save_pretrained('./saved_model')
print("✓ Model saved in SavedModel format: ./saved_model")

# Save tokenizer
tokenizer.save_pretrained('./saved_tokenizer')
print("✓ Tokenizer saved: ./saved_tokenizer")

# Save label encoder
import pickle
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("✓ Label encoder saved: label_encoder.pkl")

# Download files
from google.colab import files

print("\nDownloading model files...")
files.download('complaint_classifier_model.keras')
files.download('complaint_classifier_weights.h5')
files.download('label_encoder.pkl')

print("\n✓ All files saved and ready for download!")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 20: Model Testing (30 samples per class)
# ═══════════════════════════════════════════════════════════════

print("Testing model with 30 random samples per class...")
print("="*60)

# Prepare test samples
test_samples_per_class = 30
test_data = []

for class_idx, class_name in enumerate(label_encoder.classes_):
    class_indices = [i for i, label in enumerate(val_labels) if label == class_idx]

    if len(class_indices) >= test_samples_per_class:
        selected_indices = np.random.choice(class_indices, test_samples_per_class, replace=False)
    else:
        selected_indices = class_indices

    for idx in selected_indices:
        test_data.append({
            'text': val_texts[idx],
            'true_label': class_name,
            'true_label_idx': class_idx
        })

print(f"Total test samples: {len(test_data)}")

# Run predictions
correct_predictions = 0
results = []

for i, sample in enumerate(test_data):
    # Tokenize
    encoding = tokenizer(
        sample['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_tensors='tf'
    )

    # Predict
    prediction = model.predict(dict(encoding), verbose=0)
    predicted_idx = np.argmax(prediction.logits, axis=1)[0]
    predicted_label = label_encoder.classes_[predicted_idx]

    # Check correctness
    is_correct = (predicted_label == sample['true_label'])
    if is_correct:
        correct_predictions += 1

    results.append({
        'sample_num': i + 1,
        'text_preview': sample['text'][:100] + '...',
        'true_label': sample['true_label'],
        'predicted_label': predicted_label,
        'correct': '✓' if is_correct else '✗'
    })

    # Print every 50 samples
    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(test_data)} samples...")

# Display results
results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("TEST RESULTS SUMMARY")
print("="*60)
print(f"Total Samples Tested: {len(test_data)}")
print(f"Correct Predictions: {correct_predictions}")
print(f"Test Accuracy: {correct_predictions / len(test_data):.4f} ({correct_predictions / len(test_data) * 100:.2f}%)")

print("\n" + "="*60)
print("SAMPLE PREDICTIONS (First 20)")
print("="*60)
print(results_df.head(20).to_string(index=False))

# Show misclassifications
misclassified = results_df[results_df['correct'] == '✗']
if len(misclassified) > 0:
    print("\n" + "="*60)
    print(f"MISCLASSIFIED SAMPLES ({len(misclassified)} total)")
    print("="*60)
    print(misclassified.head(10).to_string(index=False))
else:
    print("\n✓ Perfect predictions! No misclassifications.")

print("\n" + "="*60)
print("✓ MODEL TESTING COMPLETE!")
print("="*60)